In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.southindianbank.com/interestrate/interestratelist.aspx" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find("table")
    rows=table.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    row_count = 0
    for i in rows[2:]:  #Skip heading
        data = i.find_all("td")
        #print(data)
        row=['South Indian Bank']
        row.extend([tr.text for tr in data[:3]])
        row.insert(3,'')
        row.extend([''])
        for i in range(0,5):
            row[i] = ' '.join(row[i].strip().replace('\n', '').split())
        if row_count == 19: row[1] = 'Tax-saver Fixed Deposit'
        row_count += 1
        list_of_interest_rates.append(row)
    
    #print(list_of_interest_rates)
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nSouth India Bank = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 19112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_v2_19112024.csv
            bank_name                                          tenure  \
0   South Indian Bank                               7 days to 30 days   
1   South Indian Bank                              31 days to 90 days   
2   South Indian Bank                              91 days to 99 days   
3   South Indian Bank                                        100 days   
4   South Indian Bank                            101 days to 180 days   
5   South Indian Bank                            181 days to 221 days   
6   South Indian Bank                                        222 days   
7   South Indian Bank                    223 days to less than 1 year   
8   South Indian Bank                                          1 year   
9   South Indian Bank                   Above 1 year to 1 year 6 days   
10  South Indian Bank                                   1 year 7 days   
11  South Indian Ban